# P2-G5 A2: Strict-Clean 모델링 시각화 정리 노트북

이 노트북은 새로 실행한 **P2 A비율 형성모형 strict 산출물**과 기존 **P5 major7 heterogeneity strict 산출물**을 한 곳에서 다시 읽어 lineage, 핵심 수치, 모델링 해석, 제한사항을 시각화한다.

- P2 본 실행 노트북: `workbook/p2/p2_4/p2_grade_formation_v1_strict/p2_grade_formation_strict.ipynb`
- P5 strict 산출물: `workbook/p2/p2_5/p5_major7_heterogeneity_v2_strict/`
- 이 노트북은 원천 parquet나 기존 readiness/P5 산출물을 수정하지 않고, 산출 결과만 read-only로 요약한다.
- 각 분석 셀은 `표 -> 시각화 -> 관찰/원인/제한/결론` 순서로 읽도록 구성했다.

## 0. 현시점 숫자 요약

- 실행 상태는 전체 20개 gate 중 READY/OK 계열 13개, BLOCKED 계열 1개, WARN/PENDING 계열 6개다.
- 모델링 결론의 중심은 **P2-S strict**다. P2-S 표본은 7,592행/197개 학교이고, P2-Q는 3,119행/146개 학교로 P2-S의 41.1% 규모다.
- P2-S4의 개발 R2는 12.6%지만 locked-test R2는 -6.7%이고 MAE는 10.43%p다. 즉 “설명력 있는 최종 사양”과 “holdout에서 강한 예측기”를 같은 말로 쓰면 안 된다.
- P2-Q는 `BLOCKED_FEATURE_CONTRACT`이고 blocker는 `selectivity_proxy_pct`=NOT_APPROVED(manual_approved_p4_use is not True)다. 이 branch는 현재 결론에서 제외한다.
- P5는 RAW_A strict heterogeneity는 읽을 수 있지만 residual branch는 `PENDING_UPSTREAM_RESIDUAL`이므로 P3 residual 산출 이후 재실행 대상이다.

In [1]:
from __future__ import annotations

# 공통 환경 셀이다.
# 이후 모든 셀은 여기서 고정한 P2/P5 strict artifact 경로를 read-only로 읽고,
# 생성되는 그림과 보고서만 A2 visual 폴더에 새로 저장한다.
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import platform
import warnings

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib import font_manager
from IPython.display import display, Markdown

warnings.filterwarnings("ignore", category=UserWarning)

def find_project_root(start: Path) -> Path:
    marker = Path("workbook/p2/p2_4/p2_grade_formation_v1_strict/reports/P2_GRADE_FORMATION_STATUS.json")
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / marker).exists():
            return candidate
    fallback = Path("/home/sieg/projects-wsl/SBS_dataScience")
    if (fallback / marker).exists():
        return fallback
    raise RuntimeError("project root marker not found")

PROJECT_ROOT = find_project_root(Path.cwd())

P2_ROOT = PROJECT_ROOT / "workbook/p2/p2_4/p2_grade_formation_v1_strict"
P5_ROOT = PROJECT_ROOT / "workbook/p2/p2_5/p5_major7_heterogeneity_v2_strict"
THIS_NOTEBOOK = PROJECT_ROOT / "workbook/p2/p2_5/P2_G5_A2.ipynb"
REPORT_PATH = PROJECT_ROOT / "workbook/p2/p2_5/P2_G5_A2_SUMMARY_REPORT.md"
MANIFEST_PATH = PROJECT_ROOT / "workbook/p2/p2_5/P2_G5_A2_SUMMARY_MANIFEST.json"
VISUAL_ROOT = PROJECT_ROOT / "workbook/p2/p2_5/P2_G5_A2_visuals"
FIGURES_DIR = VISUAL_ROOT / "figures"
INSIGHT_PATH = VISUAL_ROOT / "P2_G5_A2_VISUAL_INSIGHT_NOTES.md"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

def configure_korean_font() -> str:
    candidates = []
    for font_path in font_manager.findSystemFonts():
        name = Path(font_path).name.lower()
        if any(token in name for token in ["nanum", "noto", "malgun"]):
            candidates.append(font_path)
    if candidates:
        prop = font_manager.FontProperties(fname=candidates[0])
        mpl.rcParams["font.family"] = prop.get_name()
    mpl.rcParams["axes.unicode_minus"] = False
    return mpl.rcParams.get("font.family", ["default"])[0]

FONT_NAME = configure_korean_font()
plt.style.use("default")
mpl.rcParams.update({
    "figure.dpi": 130,
    "savefig.dpi": 180,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titleweight": "bold",
    "axes.grid": True,
    "grid.alpha": 0.22,
    "legend.frameon": False,
})

PALETTE = {
    "ink": "#263238",
    "muted": "#607D8B",
    "blue": "#2F6FBB",
    "teal": "#00897B",
    "green": "#43A047",
    "amber": "#F9A825",
    "red": "#C62828",
    "purple": "#6A5ACD",
    "gray": "#B0BEC5",
}
STATUS_COLORS = {
    "OK": PALETTE["green"],
    "READY": PALETTE["green"],
    "DONE": PALETTE["green"],
    "BLOCKED": PALETTE["red"],
    "WAITING": PALETTE["amber"],
    "PENDING": PALETTE["amber"],
    "WARNING": PALETTE["amber"],
    "WARN": PALETTE["amber"],
    "MODEL_CONVERGENCE_WARNING": PALETTE["amber"],
    "ERROR": PALETTE["red"],
    "MISSING": PALETTE["red"],
    "NOT_APPROVED": PALETTE["red"],
    "DENYLISTED": PALETTE["red"],
}

def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def read_json(path: Path) -> dict:
    return json.loads(path.read_text(encoding="utf-8"))

def rel(path: Path) -> str:
    try:
        return str(path.relative_to(PROJECT_ROOT))
    except ValueError:
        return str(path)

def artifact_row(path: Path, label: str) -> dict:
    return {
        "label": label,
        "path": rel(path),
        "exists": path.exists(),
        "sha256": sha256_file(path) if path.exists() else None,
        "size_bytes": path.stat().st_size if path.exists() else None,
    }

def savefig(name: str) -> str:
    path = FIGURES_DIR / name
    plt.tight_layout()
    plt.savefig(path, bbox_inches="tight")
    plt.show()
    return rel(path)

def status_bucket(status: object) -> str:
    text = str(status).upper()
    if text in {"READY", "OK", "DONE", "MATCH"}:
        return "READY/OK"
    if any(token in text for token in ["BLOCK", "DENY", "NOT_APPROVED", "MISSING", "ERROR"]):
        return "BLOCKED"
    if any(token in text for token in ["WARN", "WAIT", "PENDING", "CONVERGENCE"]):
        return "WARN/PENDING"
    if text in {"NAN", "NONE", ""}:
        return "UNKNOWN"
    return "OTHER"

def status_color(status: object) -> str:
    text = str(status).upper()
    for key, color in STATUS_COLORS.items():
        if key in text:
            return color
    return PALETTE["muted"]

def visual_note(title: str, observation: str, cause: str, limitation: str, conclusion: str) -> None:
    display(Markdown(
        f"**{title}**\n\n"
        f"- 관찰: {observation}\n"
        f"- 원인: {cause}\n"
        f"- 제한: {limitation}\n"
        f"- 결론: {conclusion}"
    ))

def add_value_labels(ax, fmt="{:.2f}", orient="v") -> None:
    if orient == "h":
        for patch in ax.patches:
            width = patch.get_width()
            ax.text(width, patch.get_y() + patch.get_height() / 2, " " + fmt.format(width), va="center", fontsize=8)
    else:
        for patch in ax.patches:
            height = patch.get_height()
            ax.text(patch.get_x() + patch.get_width() / 2, height, fmt.format(height), ha="center", va="bottom", fontsize=8)

def heatmap(ax, matrix: pd.DataFrame, title: str, cmap: str = "RdBu_r", center_zero: bool = False) -> None:
    data = matrix.astype(float)
    if center_zero:
        lim = np.nanmax(np.abs(data.to_numpy())) if data.size else 1
        im = ax.imshow(data, cmap=cmap, vmin=-lim, vmax=lim, aspect="auto")
    else:
        im = ax.imshow(data, cmap=cmap, aspect="auto")
    ax.set_title(title)
    ax.set_xticks(np.arange(data.shape[1]))
    ax.set_xticklabels(data.columns, rotation=35, ha="right")
    ax.set_yticks(np.arange(data.shape[0]))
    ax.set_yticklabels(data.index)
    for i in range(data.shape[0]):
        for j in range(data.shape[1]):
            value = data.iloc[i, j]
            if pd.notna(value):
                ax.text(j, i, f"{value:.1f}", ha="center", va="center", fontsize=8, color=PALETTE["ink"])
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

env = {
    "executed_at_utc": datetime.now(timezone.utc).isoformat(),
    "python": platform.python_version(),
    "pandas": pd.__version__,
    "numpy": np.__version__,
    "matplotlib": mpl.__version__,
    "font": FONT_NAME,
    "project_root": str(PROJECT_ROOT),
    "visual_root": rel(VISUAL_ROOT),
}
display(pd.DataFrame([env]))

fig, ax = plt.subplots(figsize=(9.5, 2.8))
ax.axis("off")
boxes = [
    (0.08, 0.58, "P2 strict\nA-rate artifacts", PALETTE["blue"]),
    (0.08, 0.18, "P5 strict\nmajor7 artifacts", PALETTE["teal"]),
    (0.48, 0.38, "P2-G5 A2\nvisual notebook", PALETTE["purple"]),
    (0.80, 0.38, "figures +\nsummary report", PALETTE["green"]),
]
for x, y, text, color in boxes:
    ax.add_patch(plt.Rectangle((x, y), 0.18, 0.22, facecolor=color, alpha=0.15, edgecolor=color, linewidth=1.8))
    ax.text(x + 0.09, y + 0.11, text, ha="center", va="center", fontsize=10, color=PALETTE["ink"])
arrowprops = dict(arrowstyle="->", lw=1.6, color=PALETTE["muted"])
ax.annotate("", xy=(0.48, 0.49), xytext=(0.26, 0.69), arrowprops=arrowprops)
ax.annotate("", xy=(0.48, 0.49), xytext=(0.26, 0.29), arrowprops=arrowprops)
ax.annotate("", xy=(0.80, 0.49), xytext=(0.66, 0.49), arrowprops=arrowprops)
ax.set_title("Read-only artifact flow", loc="left")
setup_figure = savefig("A2_00_environment_flow.png")
visual_note(
    "환경/경로 고정",
    f"프로젝트 루트와 visual 저장 경로를 고정했고 그림은 `{setup_figure}`부터 저장된다.",
    "A2는 P2/P5 산출물을 다시 계산하지 않고 현재 저장된 artifact를 읽는 요약 노트북이다.",
    "upstream artifact 자체가 바뀌면 해석도 바뀌므로 lineage hash를 다음 셀에서 확인해야 한다.",
    "이 셀 이후의 모든 그래프는 같은 root/figure 정책을 공유한다.",
)

,executed_at_utc,python,pandas,numpy,matplotlib,font,project_root,visual_root
0,2026-07-13T06:49:58.859013+00:00,3.12.3,3.0.3,2.5.0,3.11.0,NanumBarunGothic,/home/sieg/projects-wsl/SBS_dataScience,workbook/p2/p2_5/P2_G5_A2_visuals


**환경/경로 고정**

- 관찰: 프로젝트 루트와 visual 저장 경로를 고정했고 그림은 `workbook/p2/p2_5/P2_G5_A2_visuals/figures/A2_00_environment_flow.png`부터 저장된다.
- 원인: A2는 P2/P5 산출물을 다시 계산하지 않고 현재 저장된 artifact를 읽는 요약 노트북이다.
- 제한: upstream artifact 자체가 바뀌면 해석도 바뀌므로 lineage hash를 다음 셀에서 확인해야 한다.
- 결론: 이 셀 이후의 모든 그래프는 같은 root/figure 정책을 공유한다.

In [2]:
# P2/P5의 기존 산출물을 현재 파일 상태 그대로 읽는다.
# 이 셀의 상태 보드는 뒤쪽 해석에서 어떤 branch를 결론으로 쓰고 어떤 branch를 보류해야 하는지 결정한다.
p2_status = read_json(P2_ROOT / "reports/P2_GRADE_FORMATION_STATUS.json")
p2_manifest = read_json(P2_ROOT / "logs/P2_OUTPUT_MANIFEST.json")
p5_status = read_json(P5_ROOT / "reports/P5_2024_MAJOR7_HETEROGENEITY_V2_STRICT_STATUS.json")
p5_manifest = read_json(P5_ROOT / "logs/P5_OUTPUT_MANIFEST.json")

p2_sample = pd.read_csv(P2_ROOT / "qa/P2_SAMPLE_AUDIT.csv")
p2_target = pd.read_csv(P2_ROOT / "artifacts/P2_TARGET_PROFILE.csv")
p2_feature = pd.read_csv(P2_ROOT / "qa/P2_FEATURE_CONTRACT_DIFF.csv")
p2_specs = pd.read_csv(P2_ROOT / "artifacts/P2_MODEL_SPEC_REGISTRY.csv")
p2_incremental = pd.read_csv(P2_ROOT / "artifacts/P2_BLOCK_INCREMENTAL_VALUE.csv")
p2_cv = pd.read_csv(P2_ROOT / "artifacts/P2_GROUPK_FOLD_CV_METRICS.csv")
p2_test = pd.read_csv(P2_ROOT / "artifacts/P2_LOCKED_TEST_PERFORMANCE.csv")
p2_coef = pd.read_csv(P2_ROOT / "artifacts/P2_OLS_COEFFICIENTS.csv")
p2_mixed = pd.read_csv(P2_ROOT / "artifacts/P2_MIXEDLM_VARIANCE_COMPONENTS.csv")
p2_gam = pd.read_csv(P2_ROOT / "artifacts/P2_GAM_SPLINE_COMPARISON.csv")
p2_wald = pd.read_csv(P2_ROOT / "artifacts/P2_JOINT_WALD_TESTS.csv")
p2_selection = pd.read_csv(P2_ROOT / "qa/P2_SELECTIVITY_SELECTION_AUDIT.csv")

p5_slopes = pd.read_csv(P5_ROOT / "artifacts/P5_MAJOR7_SLOPE_ESTIMATES.csv")
p5_v1_compare = pd.read_csv(P5_ROOT / "artifacts/P5_V1_VS_STRICT_SENSITIVITY.csv")
p5_context = pd.read_csv(P5_ROOT / "artifacts/P5_MAJOR7_CONTEXT_PROFILE.csv")

status_rows = []
for key in [
    "P2_INPUT_STATUS", "P2_SAMPLE_STATUS", "P2_FEATURE_CONTRACT_STATUS",
    "P2_S_OLS_STATUS", "P2_Q_OLS_STATUS", "P2_GAM_STATUS",
    "P2_MIXEDLM_STATUS", "P2_SCHOOL_FE_STATUS", "P2_FRACTIONAL_STATUS",
    "P2_TEST_STATUS", "P2_TO_P3_HANDOFF_STATUS", "P2_OVERALL_STATUS",
]:
    status_rows.append({"stage": "P2", "key": key, "status": p2_status.get(key)})
for key in [
    "P5_STRICT_INPUT_STATUS", "P5_STRICT_STRUCTURE_STATUS", "P5_STRICT_SELECTIVITY_STATUS",
    "P5_RAW_A_STATUS", "P5_RESIDUAL_STATUS", "P5_CONTEXT_STATUS",
    "P5_V1_COMPARISON_STATUS", "P5_OVERALL_STATUS",
]:
    status_rows.append({"stage": "P5", "key": key, "status": p5_status.get(key)})

df_status = pd.DataFrame(status_rows)
df_status["bucket"] = df_status["status"].map(status_bucket)
display(df_status)

status_count = df_status.groupby(["stage", "bucket"]).size().unstack(fill_value=0)
ordered_cols = [c for c in ["READY/OK", "WARN/PENDING", "BLOCKED", "UNKNOWN", "OTHER"] if c in status_count.columns]
status_count = status_count.reindex(columns=ordered_cols)
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), gridspec_kw={"width_ratios": [1.05, 1.45]})
bottom = np.zeros(len(status_count))
color_map = {"READY/OK": PALETTE["green"], "WARN/PENDING": PALETTE["amber"], "BLOCKED": PALETTE["red"], "UNKNOWN": PALETTE["gray"], "OTHER": PALETTE["muted"]}
for col in status_count.columns:
    axes[0].bar(status_count.index, status_count[col], bottom=bottom, label=col, color=color_map.get(col, PALETTE["muted"]))
    bottom += status_count[col].to_numpy()
axes[0].set_title("Stage status counts")
axes[0].set_ylabel("count")
axes[0].legend(loc="upper right", fontsize=8)

plot_status = df_status.copy()
y = np.arange(len(plot_status))
axes[1].barh(y, np.ones(len(plot_status)), color=[status_color(x) for x in plot_status["status"]], alpha=0.8)
axes[1].set_yticks(y)
axes[1].set_yticklabels(plot_status["key"], fontsize=7)
axes[1].set_xticks([])
axes[1].invert_yaxis()
for i, row in plot_status.iterrows():
    axes[1].text(0.02, i, str(row["status"]), va="center", ha="left", fontsize=8, color="white" if status_bucket(row["status"]) != "WARN/PENDING" else PALETTE["ink"])
axes[1].set_title("Gate-level status board")
status_figure = savefig("A2_01_status_board.png")

blocked = df_status[df_status["bucket"].eq("BLOCKED")]
waiting = df_status[df_status["bucket"].eq("WARN/PENDING")]
visual_note(
    "상태 보드",
    f"READY/OK 단계가 {int((df_status['bucket'] == 'READY/OK').sum())}개이고, 차단/대기 단계는 {len(blocked) + len(waiting)}개다.",
    "P2-S/P5 RAW_A는 읽을 수 있지만 P2-Q와 P5 residual처럼 upstream 계약이나 잔차 산출물에 의존하는 단계가 분리돼 있다.",
    "status 문자열은 실행 상태를 요약한 것이며 성능이 좋다는 뜻은 아니다.",
    f"뒤쪽 모델링 결론은 READY/OK branch 중심으로 쓰고, BLOCKED/WARN branch는 `{status_figure}`처럼 제한으로 명시한다.",
)

,stage,key,status,bucket
0,P2,P2_INPUT_STATUS,READY,READY/OK
1,P2,P2_SAMPLE_STATUS,READY,READY/OK
2,P2,P2_FEATURE_CONTRACT_STATUS,READY_WITH_WARNINGS,WARN/PENDING
3,P2,P2_S_OLS_STATUS,READY,READY/OK
4,P2,P2_Q_OLS_STATUS,BLOCKED_FEATURE_CONTRACT,BLOCKED
5,P2,P2_GAM_STATUS,READY_WITH_WARNINGS,WARN/PENDING
6,P2,P2_MIXEDLM_STATUS,MODEL_CONVERGENCE_WARNING,WARN/PENDING
7,P2,P2_SCHOOL_FE_STATUS,READY,READY/OK
8,P2,P2_FRACTIONAL_STATUS,READY,READY/OK
9,P2,P2_TEST_STATUS,READY,READY/OK


**상태 보드**

- 관찰: READY/OK 단계가 13개이고, 차단/대기 단계는 7개다.
- 원인: P2-S/P5 RAW_A는 읽을 수 있지만 P2-Q와 P5 residual처럼 upstream 계약이나 잔차 산출물에 의존하는 단계가 분리돼 있다.
- 제한: status 문자열은 실행 상태를 요약한 것이며 성능이 좋다는 뜻은 아니다.
- 결론: 뒤쪽 모델링 결론은 READY/OK branch 중심으로 쓰고, BLOCKED/WARN branch는 `workbook/p2/p2_5/P2_G5_A2_visuals/figures/A2_01_status_board.png`처럼 제한으로 명시한다.

### 구체 수치해석: 실행 상태 보드

- READY/OK로 읽을 수 있는 단계가 13개라서 P2-S와 P5 RAW_A 중심의 요약은 가능하다.
- 차단 계열은 1개이며 핵심 차단은 P2-Q의 `BLOCKED_FEATURE_CONTRACT`다. 원인은 `selectivity_proxy_pct`가 manual-approved feature가 아니기 때문이다.
- WARN/PENDING 계열은 6개다. 여기에는 P2 GAM warning, P2 MixedLM convergence warning, P5 residual 대기가 포함된다.
- 따라서 최종 문장은 “P2/P5 전체 완료”가 아니라 **P2-S strict와 P5 RAW_A strict는 해석 가능, P2-Q/P5 residual은 보류**로 써야 한다.

## 1. Lineage와 hash

아래 표와 그래프는 본 정리 노트북이 읽은 핵심 산출물의 현재 파일 상태를 고정한다. 같은 노트북을 나중에 다시 실행했을 때 hash나 크기가 바뀌면 수치 해석도 다시 확인해야 한다.

### 구체 수치해석: lineage와 hash

- P2/P5 strict D08 SHA는 `5f56e375fd1c0474a5e55652859ae007e2f45becd6d3350ee4c82e21fab8df9b`로 일치한다. 같은 입력 기준에서 P2와 P5를 연결했다는 최소 조건은 충족한다.
- 이 셀의 파일 크기 그래프는 “무엇을 읽었는가”를 보여주는 감사판이다. 크기가 큰 notebook과 작고 결정적인 manifest/CSV를 함께 남겨야 나중에 재실행 여부를 추적할 수 있다.
- 단, 파일 크기는 내용 동일성을 보장하지 않는다. 결론 재현성은 SHA와 manifest 기준으로 확인해야 한다.

In [3]:
# strict D08는 P2/P5 모두 같은 SHA를 가리켜야 한다.
# P2/P5 notebook SHA는 각각 실행 완료 후 manifest/status에 저장된 값이다.
lineage = pd.DataFrame([
    artifact_row(P2_ROOT / "p2_grade_formation_strict.ipynb", "P2 executed notebook"),
    artifact_row(P2_ROOT / "logs/P2_OUTPUT_MANIFEST.json", "P2 output manifest"),
    artifact_row(P2_ROOT / "artifacts/P2_TO_P3_FEATURE_MATRIX_SCHEMA.json", "P2 to P3 handoff"),
    artifact_row(P5_ROOT / "p2_g5_1_strict.ipynb", "P5 strict notebook"),
    artifact_row(P5_ROOT / "logs/P5_OUTPUT_MANIFEST.json", "P5 output manifest"),
    artifact_row(P5_ROOT / "artifacts/P5_MAJOR7_SLOPE_ESTIMATES.csv", "P5 slope estimates"),
])
lineage["strict_d08_sha_from_status"] = ""
lineage.loc[0, "strict_d08_sha_from_status"] = p2_status.get("strict_d08_sha256")
lineage.loc[3, "strict_d08_sha_from_status"] = p5_status.get("strict_d08_sha256")
lineage["size_mb"] = lineage["size_bytes"].fillna(0) / (1024 * 1024)
display(lineage)

fig, ax = plt.subplots(figsize=(9.5, 3.9))
plot_df = lineage.sort_values("size_mb", ascending=True)
colors = np.where(plot_df["exists"], PALETTE["blue"], PALETTE["red"])
ax.barh(plot_df["label"], plot_df["size_mb"], color=colors, alpha=0.82)
ax.set_xlabel("file size (MB)")
ax.set_title("Current artifact footprint")
for y_pos, value in enumerate(plot_df["size_mb"]):
    ax.text(value, y_pos, f" {value:.2f} MB", va="center", fontsize=8)
lineage_figure = savefig("A2_02_lineage_artifact_sizes.png")

strict_match = p2_status.get("strict_d08_sha256") == p5_status.get("strict_d08_sha256")
visual_note(
    "Lineage/hash",
    f"P2/P5 strict D08 SHA 일치 여부는 `{strict_match}`이고, 핵심 파일 크기와 존재 여부를 `{lineage_figure}`에 저장했다.",
    "A2는 upstream notebook을 재실행하지 않고 manifest/status/CSV artifact를 읽기 때문에 파일 단위 lineage가 해석의 기준점이다.",
    "파일 크기 그래프는 내용 동등성을 보장하지 않으므로 SHA와 함께 읽어야 한다.",
    "현재 해석은 위 표의 hash와 연결된 strict-clean 산출물에 한정된다.",
)

,label,path,exists,sha256,size_bytes,strict_d08_sha_from_status,size_mb
0,P2 executed notebook,workbook/p2/p2_4/p2_grade_formation_v1_strict/...,True,1e2588b6c0d41601b513f6a891acbe2079a97e40edcf09...,303969.0,5f56e375fd1c0474a5e55652859ae007e2f45becd6d335...,0.289887
1,P2 output manifest,workbook/p2/p2_4/p2_grade_formation_v1_strict/...,True,052eeb40639fe60a23aadc3d9e7fe7c4e530baa9edcd21...,17589.0,,0.016774
2,P2 to P3 handoff,workbook/p2/p2_4/p2_grade_formation_v1_strict/...,True,2bbc7d64784b7b530d57bb6a2096d14cb11c5879f41a52...,2607.0,,0.002486
3,P5 strict notebook,workbook/p2/p2_5/p5_major7_heterogeneity_v2_st...,False,NaN,NaN,5f56e375fd1c0474a5e55652859ae007e2f45becd6d335...,0.000000
4,P5 output manifest,workbook/p2/p2_5/p5_major7_heterogeneity_v2_st...,True,7c059dce1bf5358778cf36e82566bbb2eafd23cf6052da...,20139.0,,0.019206
5,P5 slope estimates,workbook/p2/p2_5/p5_major7_heterogeneity_v2_st...,True,64205194868185d837ac25c704589d79902d22bb0cb28d...,57915.0,,0.055232


**Lineage/hash**

- 관찰: P2/P5 strict D08 SHA 일치 여부는 `True`이고, 핵심 파일 크기와 존재 여부를 `workbook/p2/p2_5/P2_G5_A2_visuals/figures/A2_02_lineage_artifact_sizes.png`에 저장했다.
- 원인: A2는 upstream notebook을 재실행하지 않고 manifest/status/CSV artifact를 읽기 때문에 파일 단위 lineage가 해석의 기준점이다.
- 제한: 파일 크기 그래프는 내용 동등성을 보장하지 않으므로 SHA와 함께 읽어야 한다.
- 결론: 현재 해석은 위 표의 hash와 연결된 strict-clean 산출물에 한정된다.

## 2. P2 표본과 feature contract

P2-S는 구조분기 전체 strict A비율 표본이고, P2-Q는 입결 관측 표본이다. 현재 P2-Q는 `selectivity_proxy_pct`가 manual-approved feature가 아니므로 계약상 차단한다.

### 구체 수치해석: 표본과 target

- P2-S는 7,592행, P2-Q는 3,119행이다. P2-Q는 P2-S의 41.1%만 남기 때문에 선택 관측 표본으로 보는 게 맞다.
- 평균 A비율은 P2-S 41.66%, P2-Q 40.76%로 P2-Q가 -0.90%p 낮다. 중앙값도 P2-Q가 -0.60%p 낮다.
- P2-S의 IQR은 32.89%~48.57%이고, P2-Q의 IQR은 33.31%~46.52%다. 두 표본의 중심은 비슷하지만 관측 메커니즘은 다르다.
- feature contract blocker는 1개이며, 현재 blocker는 `selectivity_proxy_pct`=NOT_APPROVED(manual_approved_p4_use is not True)다. 이 하나 때문에 P2-Q OLS는 실행하지 않는 것이 맞다.

In [4]:
# 표본 해석:
# row_n은 전체 eligible row 수, train/validation/test는 split audit 값이다.
# target profile은 P2-S와 P2-Q의 A비율 분포가 얼마나 비슷하거나 다른지 보여준다.
display(p2_sample)
display(p2_target)

expected_features = p2_feature[p2_feature["expected_block"].notna() & (p2_feature["expected_block"] != "")].copy()
blockers = expected_features[expected_features["status"].isin(["MISSING", "NOT_APPROVED", "DENYLISTED"])]
display(expected_features)
display(blockers)

fig, axes = plt.subplots(1, 3, figsize=(14, 4.2), gridspec_kw={"width_ratios": [1.25, 1.0, 1.0]})
split_cols = [c for c in ["train_n", "validation_n", "test_n"] if c in p2_sample.columns]
p2_sample.set_index("sample_id")[split_cols].plot(kind="bar", stacked=True, ax=axes[0], color=[PALETTE["blue"], PALETTE["amber"], PALETTE["teal"]])
axes[0].set_title("Sample split by branch")
axes[0].set_xlabel("")
axes[0].set_ylabel("rows")
axes[0].tick_params(axis="x", rotation=20)

target_plot = p2_target.set_index("branch")[["mean", "median", "p25", "p75"]]
target_plot[["mean", "median"]].plot(kind="bar", ax=axes[1], color=[PALETTE["purple"], PALETTE["green"]])
axes[1].set_title("Target center: A-rate %")
axes[1].set_xlabel("")
axes[1].set_ylabel("%")
axes[1].tick_params(axis="x", rotation=0)

blocker_counts = expected_features["status"].value_counts().rename_axis("status").reset_index(name="count")
axes[2].bar(blocker_counts["status"], blocker_counts["count"], color=[status_color(s) for s in blocker_counts["status"]])
axes[2].set_title("Feature contract status")
axes[2].set_ylabel("feature count")
axes[2].tick_params(axis="x", rotation=25)
add_value_labels(axes[2], fmt="{:.0f}")
sample_figure = savefig("A2_03_sample_feature_contract.png")

p2s_n = int(p2_sample.loc[p2_sample["sample_id"].eq("P2_STRUCTURE_READY"), "row_n"].iloc[0])
p2q_n = int(p2_sample.loc[p2_sample["sample_id"].eq("P2_SELECTIVITY_READY"), "row_n"].iloc[0])
visual_note(
    "P2 표본/feature contract",
    f"P2-S는 {p2s_n:,}행, P2-Q는 {p2q_n:,}행이며 feature blocker는 {len(blockers)}개다.",
    "P2-Q는 입결 관측 표본으로 작아지고, selectivity 관련 feature가 manual-approved 계약을 통과하지 못했다.",
    "P2-S와 P2-Q target 평균이 비슷해 보여도 관측 메커니즘이 다르므로 같은 결론으로 합치면 안 된다.",
    f"`{sample_figure}` 기준으로 P2-S는 모델링 결론, P2-Q는 추후 승인 전까지 보류 branch로 둔다.",
)

,sample_id,row_n,registry_row_n,school_n,registry_school_n,campus_n,registry_campus_n,major7_n,registry_major7_n,train_n,registry_train_n,validation_n,registry_validation_n,test_n,registry_test_n,target_non_null_n,row_id_hash
0,P2_STRUCTURE_READY,7592,7592,197,197,261,261,7,7,5534,5534,1168,1168,890,890,7592,2e3fb7557ee894b50542591376211ec84817501cd7b412...
1,P2_SELECTIVITY_READY,3119,3119,146,146,179,179,7,7,2293,2293,514,514,312,312,3119,1a0e36bb24d7e7f894897503bf0baf2c36f9b8710978b9...


,branch,n,mean,std,min,p01,p05,p25,median,p75,p95,p99,max,zero_ratio,hundred_ratio,skewness
0,P2-S,7592,41.66444,14.373115,0.0,3.556818,22.824521,32.893457,39.572113,48.571430,69.223560,83.991481,100.0,0.008825,0.002503,0.630405
1,P2-Q,3119,40.76174,12.600778,0.0,5.000000,25.857971,33.313606,38.973850,46.521971,64.928526,80.175737,100.0,0.007374,0.001603,0.722147


,feature,expected_block,actual_block,exists,manual_approved,denylisted,dtype,missing_pct_structure,missing_pct_selectivity,status,reason
0,major_group_7,S0,S0,True,True,False,str,0.000000,0.000000,MATCH,NaN
1,school_type,S0,S0,True,True,False,string,0.000000,0.000000,MATCH,NaN
2,region_sido,S0,S0,True,True,False,string,0.000000,0.000000,MATCH,NaN
3,campus_branch,S0,S0,True,True,False,string,0.000000,0.000000,MATCH,NaN
4,credit_forfeit_flag,POLICY,POLICY,True,True,False,boolean,0.000000,0.000000,MATCH,NaN
5,female_student_share_pct,B_CORE,B_CORE,True,True,False,Float32,0.004347,0.000641,MATCH,NaN
6,international_student_share_pct,B_CORE,B_CORE,True,True,False,Float32,0.004347,0.000641,MATCH,NaN
7,leave_rate_pct,B_CORE,B_CORE,True,True,False,Float32,0.004347,0.000641,MATCH,NaN
8,student_faculty_ratio,B_CORE,B_CORE,True,True,False,Float32,0.210090,0.038153,MATCH,NaN
9,fulltime_faculty_share_pct,B_CORE,B_CORE,True,True,False,Float32,0.210090,0.038153,MATCH,NaN


,feature,expected_block,actual_block,exists,manual_approved,denylisted,dtype,missing_pct_structure,missing_pct_selectivity,status,reason
12,selectivity_proxy_pct,Q_PRIMARY,Q_PRIMARY,True,False,False,Float32,0.589173,0.0,NOT_APPROVED,manual_approved_p4_use is not True


**P2 표본/feature contract**

- 관찰: P2-S는 7,592행, P2-Q는 3,119행이며 feature blocker는 1개다.
- 원인: P2-Q는 입결 관측 표본으로 작아지고, selectivity 관련 feature가 manual-approved 계약을 통과하지 못했다.
- 제한: P2-S와 P2-Q target 평균이 비슷해 보여도 관측 메커니즘이 다르므로 같은 결론으로 합치면 안 된다.
- 결론: `workbook/p2/p2_5/P2_G5_A2_visuals/figures/A2_03_sample_feature_contract.png` 기준으로 P2-S는 모델링 결론, P2-Q는 추후 승인 전까지 보류 branch로 둔다.

## 3. P2-S 중첩 OLS 설명력

`r2`는 개발표본에서 같은 P2-S 표본 안에서 계산한 설명력이다. `delta_r2_dev`는 이전 단계 대비 추가 설명력이다. `delta_cv_mae`는 양수면 MAE 개선, 음수면 악화다.

### 구체 수치해석: P2-S 중첩 OLS

- 개발 R2는 P2-S0 0.0%에서 P2-S4 12.6%로 상승한다. 가장 큰 증가는 S0 block 추가 단계이며, `delta_r2_dev`가 8.8%p 수준이다.
- B_SCALE 추가는 개발 R2를 2.22%p 올리고 CV MAE를 0.19%p 개선한다. 반면 Policy 추가는 개발 R2를 0.011%p만 올리고 CV MAE 개선값은 -0.10%p라 악화 방향이다.
- 개발 R2 기준 최고 모델은 `P2-S4`(12.6%)이지만, locked-test MAE 최저 모델은 `P2-S0`(10.02%p)다.
- P2-S4의 condition number는 13,561로 크다. 계수 방향은 읽되, 다중공선성/스케일 문제 때문에 작은 계수 차이를 과해석하지 않는다.

In [5]:
# P2-S만 실제 적합됐다. P2-Q는 feature contract 차단 상태로 남긴다.
# 개발 R2, CV, locked test를 분리해야 과적합/일반화 차이를 볼 수 있다.
p2_s_specs = p2_specs[p2_specs["branch"].eq("P2-S")].copy()
p2_s_specs["r2_pct"] = p2_s_specs["r2"] * 100
p2_s_specs["test_mae_pp"] = p2_s_specs["test_mae"]
display(p2_s_specs[[
    "model_id", "status", "N", "school_n", "r2", "r2_pct", "adj_r2",
    "condition_number", "test_mae_pp", "test_rmse", "test_r2", "block_added"
]])

p2_inc = p2_incremental[p2_incremental["branch"].eq("P2-S")].copy()
p2_inc["delta_r2_pct_point"] = p2_inc["delta_r2_dev"] * 100
display(p2_inc)
display(p2_cv)
display(p2_test)

perf = p2_s_specs.merge(p2_cv[["model_id", "cv_mae", "cv_r2"]], on="model_id", how="left")
fig, axes = plt.subplots(1, 3, figsize=(14, 4.1))
axes[0].plot(perf["model_id"], perf["r2"] * 100, marker="o", color=PALETTE["blue"], label="dev R2")
axes[0].plot(perf["model_id"], perf["test_r2"] * 100, marker="o", color=PALETTE["red"], label="locked test R2")
axes[0].axhline(0, color=PALETTE["gray"], lw=1)
axes[0].set_title("R2: dev vs locked test")
axes[0].set_ylabel("R2 (%)")
axes[0].tick_params(axis="x", rotation=25)
axes[0].legend(fontsize=8)

axes[1].plot(perf["model_id"], perf["cv_mae"], marker="o", color=PALETTE["teal"], label="CV MAE")
axes[1].plot(perf["model_id"], perf["test_mae"], marker="o", color=PALETTE["amber"], label="locked test MAE")
axes[1].set_title("MAE: CV vs locked test")
axes[1].set_ylabel("percentage points")
axes[1].tick_params(axis="x", rotation=25)
axes[1].legend(fontsize=8)

axes[2].bar(p2_inc["block_added"], p2_inc["delta_r2_pct_point"], color=PALETTE["purple"], alpha=0.82)
axes[2].axhline(0, color=PALETTE["gray"], lw=1)
axes[2].set_title("Incremental dev R2 by block")
axes[2].set_ylabel("R2 gain (pp)")
axes[2].tick_params(axis="x", rotation=25)
add_value_labels(axes[2], fmt="{:.2f}")
perf_figure = savefig("A2_04_p2_nested_ols_performance.png")

best_dev = perf.sort_values("r2", ascending=False).iloc[0]
best_test = perf.sort_values("test_mae", ascending=True).iloc[0]
visual_note(
    "P2-S 중첩 OLS",
    f"개발 R2 최고 모델은 {best_dev['model_id']}({best_dev['r2'] * 100:.1f}%)이고 locked-test MAE 최저 모델은 {best_test['model_id']}({best_test['test_mae']:.2f}%p)다.",
    "block을 추가하면 개발 설명력은 늘지만, locked-test에서는 같은 방향의 개선이 보장되지 않는다.",
    "R2와 MAE는 다른 기준이고, locked-test는 한 번만 열어보는 holdout 결과라 작은 차이를 과장하면 안 된다.",
    f"`{perf_figure}`는 P2-S 해석에서 개발 설명력과 일반화 성능을 분리해야 한다는 핵심 근거다.",
)

,model_id,status,N,school_n,r2,r2_pct,adj_r2,condition_number,test_mae_pp,test_rmse,test_r2,block_added
0,P2-S0,READY,6702.0,167.0,1.110223e-16,1.110223e-14,1.110223e-16,1.000000,10.019603,13.650441,-0.002345,intercept
1,P2-S1,READY,6702.0,167.0,8.842039e-02,8.842039e+00,8.473255e-02,273.216474,10.309273,14.174188,-0.080738,S0
2,P2-S2,READY,6702.0,167.0,1.041557e-01,1.041557e+01,9.958712e-02,13522.850735,10.363653,14.258991,-0.093708,B_CORE
3,P2-S3,READY,6702.0,167.0,1.263332e-01,1.263332e+01,1.216142e-01,13560.670539,10.426699,14.085459,-0.067249,B_SCALE
4,P2-S4,READY,6702.0,167.0,1.264461e-01,1.264461e+01,1.215960e-01,13561.076066,10.432357,14.082241,-0.066762,POLICY


,branch,from_model,to_model,block_added,delta_r2_dev,delta_cv_mae,delta_cv_r2,status,delta_r2_pct_point
0,P2-S,P2-S0,P2-S1,S0,0.088420,0.154054,0.030860,READY,8.842039
1,P2-S,P2-S1,P2-S2,B_CORE,0.015735,0.057888,0.012187,READY,1.573531
2,P2-S,P2-S2,P2-S3,B_SCALE,0.022177,0.190278,0.028122,READY,2.217746
3,P2-S,P2-S3,P2-S4,Policy,0.000113,-0.097367,-0.013531,READY,0.011298


,model_id,cv_n,cv_mae,cv_rmse,cv_r2
0,P2-S0,6702,10.777899,14.476274,-0.001485
1,P2-S1,6702,10.623845,14.251494,0.029375
2,P2-S2,6702,10.565957,14.161743,0.041561
3,P2-S3,6702,10.375679,13.952432,0.069684
4,P2-S4,6702,10.473046,14.053527,0.056153


,model_id,branch,test_n,test_mae,test_rmse,test_r2,test_usage
0,P2-S0,P2-S,890,10.019603,13.650441,-0.002345,locked_once_after_spec_definition
1,P2-S1,P2-S,890,10.309273,14.174188,-0.080738,locked_once_after_spec_definition
2,P2-S2,P2-S,890,10.363653,14.258991,-0.093708,locked_once_after_spec_definition
3,P2-S3,P2-S,890,10.426699,14.085459,-0.067249,locked_once_after_spec_definition
4,P2-S4,P2-S,890,10.432357,14.082241,-0.066762,locked_once_after_spec_definition


**P2-S 중첩 OLS**

- 관찰: 개발 R2 최고 모델은 P2-S4(12.6%)이고 locked-test MAE 최저 모델은 P2-S0(10.02%p)다.
- 원인: block을 추가하면 개발 설명력은 늘지만, locked-test에서는 같은 방향의 개선이 보장되지 않는다.
- 제한: R2와 MAE는 다른 기준이고, locked-test는 한 번만 열어보는 holdout 결과라 작은 차이를 과장하면 안 된다.
- 결론: `workbook/p2/p2_5/P2_G5_A2_visuals/figures/A2_04_p2_nested_ols_performance.png`는 P2-S 해석에서 개발 설명력과 일반화 성능을 분리해야 한다는 핵심 근거다.

## 4. P2-S 핵심 계수와 수치 해석

계수 단위는 A비율 percentage point다. 예를 들어 `log_graduates` 계수 1.62는 다른 공변량을 조건화했을 때 `log1p(graduates_n)` 1단위 증가와 A비율 약 1.62%p 증가가 연결됐다는 뜻이다. 인과효과로 해석하지 않는다.

### 구체 수치해석: P2-S4 계수

- 핵심 연속/정책 변수 8개 중 95% CI가 0을 제외한 항목은 3개다. 대표적으로 `log_graduates`는 계수 1.635%p, CI [1.245, 2.024], 표준화 beta 0.185로 양의 방향이다.
- `log_enrolled_students`는 계수 -2.105%p, CI [-2.954, -1.256], 표준화 beta -0.141로 음의 방향이다. 같은 규모 변수라도 졸업생 규모와 재학생 규모가 반대 방향으로 잡힌다.
- `fulltime_faculty_share_pct`는 1%p 증가당 A비율 -0.049%p 낮은 방향이며 CI가 0을 제외한다. 다만 이것은 조건부 association이지 교원 비중을 늘리면 A비율이 내려간다는 인과문장이 아니다.
- `female_student_share_pct`는 계수 0.052%p, p=0.0602로 경계적이고, `credit_forfeit_flag_True`는 계수 -0.585%p지만 CI가 [-7.575, 6.405]라 0을 넓게 포함한다.
- 절대 표준화 beta 상위 5개는 `log_graduates` β=0.185, `school_type_교육대학` β=-0.159, `region_sido_서울` β=0.147, `log_enrolled_students` β=-0.141, `major_group_7_SOC` β=-0.134다. 이 목록은 “상대적으로 큰 조건부 연관”이지 정책 우선순위가 아니다.

In [6]:
# 핵심 수치형/정책 계수만 따로 뽑아 사람이 읽을 수 있는 해석 문장을 붙인다.
# forest plot은 계수 방향과 CI가 0을 지나는지를 동시에 보여준다.
term_labels = {
    "female_student_share_pct": "여학생 비중",
    "international_student_share_pct": "외국인 학생 비중",
    "leave_rate_pct": "중도탈락률",
    "student_faculty_ratio": "학생-교원비",
    "fulltime_faculty_share_pct": "전임교원 비중",
    "log_enrolled_students": "재학생 규모 log1p",
    "log_graduates": "졸업생 규모 log1p",
    "credit_forfeit_flag_True": "학점포기 제도 확인",
}
coef_focus = p2_coef[(p2_coef["model_id"].eq("P2-S4")) & (p2_coef["term"].isin(term_labels))].copy()
coef_focus["label"] = coef_focus["term"].map(term_labels)
coef_focus["direction"] = np.where(coef_focus["coefficient"] > 0, "positive", "negative")
coef_focus["ci_excludes_zero"] = ~((coef_focus["ci_low"] <= 0) & (coef_focus["ci_high"] >= 0))
coef_focus["interpretation"] = coef_focus.apply(
    lambda r: f"{r['label']}: 계수 {r['coefficient']:.3f}%p, 95% CI [{r['ci_low']:.3f}, {r['ci_high']:.3f}], {'0을 포함하지 않음' if r['ci_excludes_zero'] else '0을 포함'}",
    axis=1,
)
display(coef_focus[[
    "label", "term", "coefficient", "school_clustered_se", "ci_low", "ci_high",
    "p_value", "standardized_beta", "direction", "ci_excludes_zero", "interpretation",
]])

s4_top = p2_coef[(p2_coef["model_id"].eq("P2-S4")) & (~p2_coef["term"].eq("Intercept"))].copy()
s4_top["abs_standardized_beta"] = s4_top["standardized_beta"].abs()
s4_top = s4_top.sort_values("abs_standardized_beta", ascending=False)
display(s4_top.head(15)[[
    "term", "coefficient", "school_clustered_se", "ci_low", "ci_high", "p_value", "standardized_beta",
]])

fig, axes = plt.subplots(1, 2, figsize=(14, 5.2), gridspec_kw={"width_ratios": [1.2, 1.0]})
forest = coef_focus.sort_values("coefficient")
y = np.arange(len(forest))
xerr = np.vstack([forest["coefficient"] - forest["ci_low"], forest["ci_high"] - forest["coefficient"]])
axes[0].errorbar(forest["coefficient"], y, xerr=xerr, fmt="o", color=PALETTE["blue"], ecolor=PALETTE["gray"], capsize=3)
axes[0].axvline(0, color=PALETTE["red"], lw=1)
axes[0].set_yticks(y)
axes[0].set_yticklabels(forest["label"])
axes[0].set_xlabel("coefficient (%p)")
axes[0].set_title("P2-S4 key coefficients with 95% CI")

top_beta = s4_top.head(12).sort_values("standardized_beta")
axes[1].barh(top_beta["term"], top_beta["standardized_beta"], color=np.where(top_beta["standardized_beta"] >= 0, PALETTE["green"], PALETTE["red"]), alpha=0.78)
axes[1].axvline(0, color=PALETTE["gray"], lw=1)
axes[1].set_title("Top standardized beta terms")
axes[1].set_xlabel("standardized beta")
coef_figure = savefig("A2_05_p2_coefficients_forest.png")

nonzero_focus = coef_focus[coef_focus["ci_excludes_zero"]]
strongest = s4_top.dropna(subset=["standardized_beta"]).iloc[0]
visual_note(
    "P2-S 핵심 계수",
    f"핵심 수치형/정책 계수 중 95% CI가 0을 제외한 항목은 {len(nonzero_focus)}개이고, 절대 표준화 beta 최대 항은 `{strongest['term']}`이다.",
    "forest plot은 조건부 연관의 방향과 불확실성을 같이 보여주며, standardized beta는 변수 단위 차이를 줄여 상대 크기를 비교하게 한다.",
    "범주형 더미와 연속형 변수의 beta를 같은 실질 효과로 읽으면 안 되고, 계수는 인과효과가 아니다.",
    f"`{coef_figure}` 기준으로 해석은 방향/CI/상대크기 세 층으로 분리한다.",
)

,label,term,coefficient,school_clustered_se,ci_low,ci_high,p_value,standardized_beta,direction,ci_excludes_zero,interpretation
129,학점포기 제도 확인,credit_forfeit_flag_True,-0.585319,3.566427,-7.575387,6.404748,8.696372e-01,-0.013147,negative,False,"학점포기 제도 확인: 계수 -0.585%p, 95% CI [-7.575, 6.405..."
130,여학생 비중,female_student_share_pct,0.051953,0.027644,-0.002228,0.106134,6.019242e-02,0.091613,positive,False,"여학생 비중: 계수 0.052%p, 95% CI [-0.002, 0.106], 0을 포함"
132,외국인 학생 비중,international_student_share_pct,0.002829,0.023986,-0.044182,0.049841,9.060953e-01,0.002365,positive,False,"외국인 학생 비중: 계수 0.003%p, 95% CI [-0.044, 0.050],..."
133,중도탈락률,leave_rate_pct,0.034752,0.030876,-0.025764,0.095268,2.603633e-01,0.031027,positive,False,"중도탈락률: 계수 0.035%p, 95% CI [-0.026, 0.095], 0을 포함"
134,학생-교원비,student_faculty_ratio,-0.011590,0.011331,-0.033798,0.010618,3.063631e-01,-0.030162,negative,False,"학생-교원비: 계수 -0.012%p, 95% CI [-0.034, 0.011], 0..."
136,전임교원 비중,fulltime_faculty_share_pct,-0.048790,0.012291,-0.072880,-0.024701,7.197670e-05,-0.079117,negative,True,"전임교원 비중: 계수 -0.049%p, 95% CI [-0.073, -0.025],..."
137,재학생 규모 log1p,log_enrolled_students,-2.105105,0.433151,-2.954066,-1.256144,1.173996e-06,-0.141256,negative,True,"재학생 규모 log1p: 계수 -2.105%p, 95% CI [-2.954, -1...."
138,졸업생 규모 log1p,log_graduates,1.634815,0.198815,1.245144,2.024485,1.988287e-16,0.185390,positive,True,"졸업생 규모 log1p: 계수 1.635%p, 95% CI [1.245, 2.024..."


,term,coefficient,school_clustered_se,ci_low,ci_high,p_value,standardized_beta
138,log_graduates,1.634815,0.198815,1.245144,2.024485,1.988287e-16,0.185390
108,school_type_교육대학,-18.155782,3.108769,-24.248858,-12.062706,5.214346e-09,-0.158756
119,region_sido_서울,5.337631,3.588319,-1.695346,12.370607,1.368824e-01,0.147123
137,log_enrolled_students,-2.105105,0.433151,-2.954066,-1.256144,1.173996e-06,-0.141256
107,major_group_7_SOC,-4.814786,1.087138,-6.945538,-2.684034,9.473156e-06,-0.134082
130,female_student_share_pct,0.051953,0.027644,-0.002228,0.106134,6.019242e-02,0.091613
106,major_group_7_NAT,-3.859061,1.169753,-6.151735,-1.566386,9.701672e-04,-0.089399
136,fulltime_faculty_share_pct,-0.048790,0.012291,-0.072880,-0.024701,7.197670e-05,-0.079117
103,major_group_7_ENG,-2.421634,1.221407,-4.815549,-0.027720,4.740553e-02,-0.072910
122,region_sido_인천,6.395382,2.253369,1.978859,10.811904,4.537701e-03,0.070893


**P2-S 핵심 계수**

- 관찰: 핵심 수치형/정책 계수 중 95% CI가 0을 제외한 항목은 3개이고, 절대 표준화 beta 최대 항은 `log_graduates`이다.
- 원인: forest plot은 조건부 연관의 방향과 불확실성을 같이 보여주며, standardized beta는 변수 단위 차이를 줄여 상대 크기를 비교하게 한다.
- 제한: 범주형 더미와 연속형 변수의 beta를 같은 실질 효과로 읽으면 안 되고, 계수는 인과효과가 아니다.
- 결론: `workbook/p2/p2_5/P2_G5_A2_visuals/figures/A2_05_p2_coefficients_forest.png` 기준으로 해석은 방향/CI/상대크기 세 층으로 분리한다.

## 5. P2 비선형성, 학교 분산, 고정효과 감도

GAM/spline의 `delta_aic_spline_minus_linear`가 음수면 spline 사양의 AIC가 더 낮다. MixedLM의 ICC는 A비율 분산 중 학교 간 차이와 연결된 비율이지만, 현재 조정모형은 convergence warning을 동반하므로 보수적으로 읽는다.

### 구체 수치해석: 비선형성, 학교분산, 공동검정

- GAM에서 spline AIC가 선형보다 낮은 변수는 `student_faculty_ratio`, `log_graduates`다. 가장 큰 개선은 `student_faculty_ratio`이며 `spline AIC - linear AIC = -23.56`다.
- `log_enrolled_students`는 delta AIC가 0.068로 거의 0이어서 spline을 굳이 쓸 근거가 약하다.
- MixedLM 조정모형 ICC는 33.8%다. A비율 변동 중 학교 단위로 묶이는 성분이 작지 않다는 뜻이지만, 상태가 `MODEL_CONVERGENCE_WARNING`라 추정 안정성 경고를 함께 써야 한다.
- P2-S4 Wald 검정에서 가장 강한 block은 `region_sido` p=5.93e-273, `school_type` p=1.14e-08, `major_group_7` p=4.23e-05다. 반대로 약한 block은 `credit_forfeit_flag` p=0.8696, `campus_branch` p=0.3117다.

In [7]:
# 이 셀은 선형계수표만으로 놓치기 쉬운 세 가지 진단을 묶는다.
# 1) spline이 선형보다 나은지, 2) 학교 간 분산이 남는지, 3) block 공동검정이 강한지를 본다.
gam_read = p2_gam.copy()
gam_read["aic_reading"] = gam_read.apply(
    lambda r: "spline lower AIC" if pd.notna(r.get("delta_aic_spline_minus_linear")) and r["delta_aic_spline_minus_linear"] < 0 else (
        "linear not worse by AIC" if pd.notna(r.get("delta_aic_spline_minus_linear")) else r["status"]
    ),
    axis=1,
)
display(gam_read)

mixed_read = p2_mixed.copy()
mixed_read["icc_pct"] = mixed_read["icc"] * 100
display(mixed_read)
wald_s4 = p2_wald[p2_wald["model_id"].eq("P2-S4")].copy()
wald_s4["neg_log10_p"] = -np.log10(wald_s4["p_value"].clip(lower=1e-300))
display(wald_s4)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.4))
gam_plot = gam_read.sort_values("delta_aic_spline_minus_linear")
axes[0].barh(gam_plot["variable"], gam_plot["delta_aic_spline_minus_linear"], color=np.where(gam_plot["delta_aic_spline_minus_linear"] < 0, PALETTE["green"], PALETTE["amber"]))
axes[0].axvline(0, color=PALETTE["gray"], lw=1)
axes[0].set_title("GAM: spline AIC - linear AIC")
axes[0].set_xlabel("negative = spline lower AIC")

axes[1].bar(mixed_read["model"].fillna("NULL"), mixed_read["icc_pct"], color=[status_color(s) for s in mixed_read["status"]], alpha=0.82)
axes[1].set_title("MixedLM ICC")
axes[1].set_ylabel("school-level share (%)")
axes[1].tick_params(axis="x", rotation=25)
add_value_labels(axes[1], fmt="{:.1f}")

wald_plot = wald_s4.sort_values("neg_log10_p", ascending=True)
axes[2].barh(wald_plot["block"], wald_plot["neg_log10_p"], color=PALETTE["purple"], alpha=0.82)
axes[2].axvline(-np.log10(0.05), color=PALETTE["red"], ls="--", lw=1)
axes[2].set_title("P2-S4 joint Wald tests")
axes[2].set_xlabel("-log10(p-value)")
diag_figure = savefig("A2_06_p2_nonlinearity_variance_tests.png")

spline_better = gam_read[gam_read["delta_aic_spline_minus_linear"] < 0]["variable"].tolist()
adjusted_icc = float(mixed_read.loc[mixed_read["model"].eq("ADJUSTED_S4"), "icc_pct"].iloc[0]) if mixed_read["model"].eq("ADJUSTED_S4").any() else np.nan
visual_note(
    "비선형성/학교분산/공동검정",
    f"spline AIC가 더 낮은 변수는 {', '.join(spline_better) if spline_better else '없음'}이고, 조정 MixedLM ICC는 {adjusted_icc:.1f}%다.",
    "일부 구조 변수는 선형 하나로 요약하기 어렵고, 학교 단위 분산도 조정 후 남아 있다.",
    "MixedLM warning과 p-value 극단값은 모델 안정성/표본 구조의 영향을 받으므로 방향성 진단으로 읽는다.",
    f"`{diag_figure}`는 P2-S 계수 해석 뒤에 붙는 모델 진단 보드다.",
)

,variable,model_id,status,linear_aic,spline_aic,delta_aic_spline_minus_linear,effective_degrees_of_freedom,figure,branch,aic_reading
0,student_faculty_ratio,P2-S4,READY,54001.831575,53978.267455,-23.564119,5.0,P2_GAM_P2-S4_student_faculty_ratio.png,P2-S,spline lower AIC
1,log_enrolled_students,P2-S4,READY,54001.831575,54001.899359,0.067785,5.0,P2_GAM_P2-S4_log_enrolled_students.png,P2-S,linear not worse by AIC
2,log_graduates,P2-S4,READY,54001.831575,53980.418282,-21.413293,5.0,P2_GAM_P2-S4_log_graduates.png,P2-S,spline lower AIC
3,selectivity_proxy_pct,P2-Q4,BLOCKED_FEATURE_CONTRACT,NaN,NaN,NaN,NaN,NaN,P2-Q,BLOCKED_FEATURE_CONTRACT
4,competition_ratio,P2-Q4,BLOCKED_FEATURE_CONTRACT,NaN,NaN,NaN,NaN,NaN,P2-Q,BLOCKED_FEATURE_CONTRACT


,model,school_variance,residual_variance,icc,log_likelihood,aic,bic,converged,status,icc_pct
0,NaN,0.000000,141.098572,0.000000,inf,-inf,-inf,True,READY,0.000000
1,ADJUSTED_S4,66.480558,130.080932,0.338218,-2.604509e+04,5.217018e+04,5.244259e+04,True,MODEL_CONVERGENCE_WARNING,33.821761


,model_id,block,df,wald_stat,p_value,neg_log10_p
12,P2-S4,major_group_7,6,29.834756,4.225604e-05,4.374111
13,P2-S4,school_type,4,42.789937,1.144020e-08,7.941566
14,P2-S4,region_sido,16,1327.594446,5.928482e-273,272.227056
15,P2-S4,campus_branch,1,1.023423,3.117085e-01,0.506251
16,P2-S4,credit_forfeit_flag,1,0.026935,8.696372e-01,0.060662


**비선형성/학교분산/공동검정**

- 관찰: spline AIC가 더 낮은 변수는 student_faculty_ratio, log_graduates이고, 조정 MixedLM ICC는 33.8%다.
- 원인: 일부 구조 변수는 선형 하나로 요약하기 어렵고, 학교 단위 분산도 조정 후 남아 있다.
- 제한: MixedLM warning과 p-value 극단값은 모델 안정성/표본 구조의 영향을 받으므로 방향성 진단으로 읽는다.
- 결론: `workbook/p2/p2_5/P2_G5_A2_visuals/figures/A2_06_p2_nonlinearity_variance_tests.png`는 P2-S 계수 해석 뒤에 붙는 모델 진단 보드다.

## 6. 입결 관측 선택편향 감사

P2-Q는 계약상 차단됐지만, 입결 관측 표본이 전체 strict 표본과 어떻게 다른지는 QA로 남긴다. 이는 나중에 `selectivity_proxy_pct`가 feature로 승인될 때 P2-Q 해석 범위를 정하는 근거다.

### 구체 수치해석: 입결 관측 선택편향

- 입결 관측 표본(`has_selectivity=True`)의 평균 A비율은 40.76%, 미관측 표본은 42.29%다. 차이는 -1.53%p로 관측 표본이 더 낮다.
- 관측 표본은 평균 `log_enrolled_students`가 0.615 높다. log 차이를 단순 지수화하면 약 1.85배 규모 차이에 해당한다.
- 평균 `log_graduates`도 0.206 높지만, `student_faculty_ratio`는 -4.40%p 낮다. 즉 입결 관측 여부는 단순 결측이 아니라 학교/전공 구조 차이를 동반한다.
- 그래서 P2-Q가 차단된 상태에서 P2-S 결론을 P2-Q 표본에 그대로 일반화하면 안 된다.

In [8]:
# has_selectivity=True/False는 관측 여부에 따른 표본 차이를 보여준다.
# 이 차이가 크면 P2-Q 모델은 '입결 관측 가능한 학교/전공'에 대한 조건부 해석으로 제한된다.
selection_numeric = p2_selection[p2_selection["variable"].isin([
    "a_rate_pct", "log_enrolled_students", "log_graduates", "student_faculty_ratio"
])].copy()
selection_numeric["has_selectivity_label"] = selection_numeric["has_selectivity"].astype(str)
display(selection_numeric)

pivot = selection_numeric.pivot_table(index="variable", columns="has_selectivity_label", values="mean", aggfunc="first")
for col in ["False", "True"]:
    if col not in pivot.columns:
        pivot[col] = np.nan
pivot["diff_true_minus_false"] = pivot["True"] - pivot["False"]
display(pivot)

a_no = pivot.loc["a_rate_pct", "False"]
a_yes = pivot.loc["a_rate_pct", "True"]
print(f"입결 관측 표본의 평균 A비율은 {a_yes:.2f}%이고, 미관측 표본은 {a_no:.2f}%이다. 차이는 {a_yes - a_no:.2f}%p다.")

fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.3), gridspec_kw={"width_ratios": [1.2, 1.0]})
pivot[["False", "True"]].plot(kind="bar", ax=axes[0], color=[PALETTE["gray"], PALETTE["blue"]])
axes[0].set_title("Means by selectivity availability")
axes[0].set_xlabel("")
axes[0].set_ylabel("mean")
axes[0].tick_params(axis="x", rotation=25)
axes[0].legend(title="has_selectivity", fontsize=8)

diff_plot = pivot.sort_values("diff_true_minus_false")
axes[1].barh(diff_plot.index, diff_plot["diff_true_minus_false"], color=np.where(diff_plot["diff_true_minus_false"] >= 0, PALETTE["green"], PALETTE["red"]), alpha=0.8)
axes[1].axvline(0, color=PALETTE["gray"], lw=1)
axes[1].set_title("True - False mean difference")
axes[1].set_xlabel("difference")
selection_figure = savefig("A2_07_selectivity_bias_audit.png")

visual_note(
    "입결 관측 선택편향",
    f"입결 관측 표본의 평균 A비율은 미관측 표본보다 {a_yes - a_no:.2f}%p 낮다.",
    "입결 자료가 관측되는 학교/전공은 무작위 subset이 아니며 규모·교원비·졸업생 구조도 같이 달라질 수 있다.",
    "이 감사는 선택편향 가능성을 보여줄 뿐 보정된 P2-Q 모델 결과를 제공하지 않는다.",
    f"`{selection_figure}` 때문에 P2-Q는 feature 승인과 선택모형 설계 전까지 결론에서 제외한다.",
)

,variable,has_selectivity,n,mean,median,missing_pct,level,share_no_selectivity,share_has_selectivity,has_selectivity_label
0,a_rate_pct,False,4473.0,42.293884,40.219994,0.000000,NaN,NaN,NaN,False
1,a_rate_pct,True,3119.0,40.761742,38.973850,0.000000,NaN,NaN,NaN,True
2,log_enrolled_students,False,4473.0,4.638486,4.736198,0.000000,NaN,NaN,NaN,False
3,log_enrolled_students,True,3119.0,5.253861,5.273000,0.000000,NaN,NaN,NaN,True
4,log_graduates,False,4473.0,2.604549,3.178054,0.000000,NaN,NaN,NaN,False
5,log_graduates,True,3119.0,2.811035,3.433987,0.000000,NaN,NaN,NaN,True
6,student_faculty_ratio,False,4473.0,22.109516,13.166667,0.329980,NaN,NaN,NaN,False
7,student_faculty_ratio,True,3119.0,17.708256,14.658333,0.038153,NaN,NaN,NaN,True


has_selectivity_label,False,True,diff_true_minus_false
variable,,,
a_rate_pct,42.293884,40.761742,-1.532143
log_enrolled_students,4.638486,5.253861,0.615375
log_graduates,2.604549,2.811035,0.206486
student_faculty_ratio,22.109516,17.708256,-4.401260


입결 관측 표본의 평균 A비율은 40.76%이고, 미관측 표본은 42.29%이다. 차이는 -1.53%p다.


**입결 관측 선택편향**

- 관찰: 입결 관측 표본의 평균 A비율은 미관측 표본보다 -1.53%p 낮다.
- 원인: 입결 자료가 관측되는 학교/전공은 무작위 subset이 아니며 규모·교원비·졸업생 구조도 같이 달라질 수 있다.
- 제한: 이 감사는 선택편향 가능성을 보여줄 뿐 보정된 P2-Q 모델 결과를 제공하지 않는다.
- 결론: `workbook/p2/p2_5/P2_G5_A2_visuals/figures/A2_07_selectivity_bias_audit.png` 때문에 P2-Q는 feature 승인과 선택모형 설계 전까지 결론에서 제외한다.

## 7. 기존 P5 strict 산출물 요약

P5는 이미 strict D08와 strict sample membership에서 실행된 산출물을 읽는다. 현재 실행 signal은 RAW_A이고, residual branch는 P3 upstream residual 대기 상태다.

### 구체 수치해석: P5 strict major7 heterogeneity

- 진학률(`GRAD_SCHOOL_PROGRESSION`)은 7개 major 모두 AME가 양수다. 중앙값은 1.17%p, 최대는 `ENG` 4.57%p, 최소는 `MED` 0.38%p다.
- 진학률에서 CI 하한이 0보다 큰 major는 6/7개다. `MED`는 AME가 양수지만 CI 하한이 -0.06%p라 0 포함 가능성을 남긴다.
- 건강취업률(`HEALTH_EMPLOYMENT`)은 양수 major 4개, 음수 major 3개다. 최대는 `HUM` 2.05%p, 최소는 `MED` -0.57%p다.
- 건강취업률에서 CI 하한이 0보다 큰 major는 3/7개다. 따라서 P5 결론은 “A비율 효과가 전 계열에서 동일”이 아니라 **outcome과 major별로 방향/불확실성이 갈린다**가 맞다.
- V1-vs-strict 비교에서 sign change는 0건이고 최대 절대변화도 8.20e-15%p 수준이다. 현재 strict 전환은 P5 primary RAW_A 방향을 사실상 바꾸지 않았다.

In [9]:
# P5 주 해석은 STRUCTURE/B_CORE/fractional_logit/RAW_A의 AME다.
# AME는 확률 단위라서 여기서는 100을 곱해 percentage point로 표시한다.
p5_primary = p5_slopes[
    p5_slopes["branch"].eq("STRUCTURE")
    & p5_slopes["control_set"].eq("B_CORE")
    & p5_slopes["model_family"].eq("fractional_logit")
    & p5_slopes["grade_signal"].eq("RAW_A")
    & p5_slopes["primary_model"].astype(bool)
].copy()
p5_primary["ame_pp_10pp"] = p5_primary["ame_10pp"] * 100
p5_primary["ci_low_pp_10pp"] = p5_primary["ame_ci_low_10pp"] * 100
p5_primary["ci_high_pp_10pp"] = p5_primary["ame_ci_high_10pp"] * 100

p5_summary = (
    p5_primary.groupby("outcome")
    .agg(
        major_n=("major_group_7", "nunique"),
        positive_n=("ame_pp_10pp", lambda s: int((s > 0).sum())),
        median_ame_pp=("ame_pp_10pp", "median"),
        min_ame_pp=("ame_pp_10pp", "min"),
        max_ame_pp=("ame_pp_10pp", "max"),
    )
    .reset_index()
)
display(p5_summary)
display(p5_primary[[
    "major_group_7", "outcome", "row_n", "school_n",
    "ame_pp_10pp", "ci_low_pp_10pp", "ci_high_pp_10pp", "status", "model_warning",
]].sort_values(["outcome", "major_group_7"]))

# V1 vs strict sensitivity는 같은 행 수로 비교된 경우가 많다. sign_change가 있는지 우선 확인한다.
v1_primary = p5_v1_compare[
    p5_v1_compare["branch"].eq("STRUCTURE")
    & p5_v1_compare["control_set"].eq("B_CORE")
    & p5_v1_compare["model_family"].eq("fractional_logit")
    & p5_v1_compare["grade_signal"].eq("RAW_A")
].copy()
v1_primary["absolute_change_pp"] = v1_primary["absolute_change"] * 100
display(v1_primary)
print("P5 strict residual status:", p5_status.get("P5_RESIDUAL_STATUS"))

fig, axes = plt.subplots(1, 2, figsize=(13.5, 4.8), gridspec_kw={"width_ratios": [1.2, 1.0]})
ame_matrix = p5_primary.pivot_table(index="major_group_7", columns="outcome", values="ame_pp_10pp", aggfunc="first").sort_index()
heatmap(axes[0], ame_matrix, "P5 RAW_A AME per +10pp A-rate", center_zero=True)

sens = v1_primary.pivot_table(index="major_group_7", columns="outcome", values="absolute_change_pp", aggfunc="first").reindex(ame_matrix.index)
if sens.notna().any().any():
    heatmap(axes[1], sens.fillna(0), "V1 vs strict absolute change (pp)", cmap="YlGnBu", center_zero=False)
else:
    axes[1].axis("off")
    axes[1].text(0.5, 0.5, "No sensitivity rows", ha="center", va="center")
p5_figure = savefig("A2_08_p5_strict_heterogeneity.png")

pos_by_outcome = p5_summary.set_index("outcome")["positive_n"].to_dict()
sign_changes = int(v1_primary["sign_change"].fillna(False).sum()) if not v1_primary.empty else 0
visual_note(
    "P5 strict heterogeneity",
    f"outcome별 양의 AME major 수는 {pos_by_outcome}이고, V1-vs-strict sign change는 {sign_changes}건이다.",
    "major7별 RAW_A 기울기는 outcome에 따라 방향과 크기가 다르며, strict 전환 민감도는 별도 축으로 확인해야 한다.",
    "AME는 조건부 association이고 residual branch는 아직 P3 residual 산출물 대기 상태다.",
    f"`{p5_figure}`는 P5를 하나의 평균효과가 아니라 major/outcome별 이질성 문제로 읽게 한다.",
)

,outcome,major_n,positive_n,median_ame_pp,min_ame_pp,max_ame_pp
0,GRAD_SCHOOL_PROGRESSION,7,7,1.169029,0.380108,4.569829
1,HEALTH_EMPLOYMENT,7,4,0.714446,-0.565665,2.045648


,major_group_7,outcome,row_n,school_n,ame_pp_10pp,ci_low_pp_10pp,ci_high_pp_10pp,status,model_warning
2,ART,GRAD_SCHOOL_PROGRESSION,646,138,0.617426,0.051498,1.680044,OK,school_type: rare_levels=2 | region_sido: rare...
6,EDU,GRAD_SCHOOL_PROGRESSION,602,117,0.531553,0.164020,1.211736,OK,school_type: rare_levels=2 | campus_branch: ra...
10,ENG,GRAD_SCHOOL_PROGRESSION,911,130,4.569829,1.622846,9.121355,OK,school_type: rare_levels=3
14,HUM,GRAD_SCHOOL_PROGRESSION,535,106,1.373239,0.409041,3.164572,OK,school_type: rare_levels=1 | region_sido: rare...
18,MED,GRAD_SCHOOL_PROGRESSION,374,115,0.380108,-0.059264,7.307119,OK,school_type: rare_levels=2 | region_sido: rare...
22,NAT,GRAD_SCHOOL_PROGRESSION,583,111,3.673025,1.168945,6.987563,OK,school_type: rare_levels=2
26,SOC,GRAD_SCHOOL_PROGRESSION,848,148,1.169029,0.517430,2.329865,OK,school_type: rare_levels=2
0,ART,HEALTH_EMPLOYMENT,646,138,-0.429911,-1.468160,0.732501,OK,school_type: rare_levels=2 | region_sido: rare...
4,EDU,HEALTH_EMPLOYMENT,602,117,1.563349,0.025616,3.102570,OK,school_type: rare_levels=2 | campus_branch: ra...
8,ENG,HEALTH_EMPLOYMENT,911,130,1.929391,0.827751,2.875462,OK,school_type: rare_levels=3


,major_group_7,outcome,model_family,old_n,strict_n,old_ame_10pp,strict_ame_10pp,absolute_change,sign_change,old_ci_low,old_ci_high,strict_ci_low,strict_ci_high,ci_overlap,branch,control_set,grade_signal,absolute_change_pp
28,ART,GRAD_SCHOOL_PROGRESSION,fractional_logit,646,646,0.006174,0.006174,1.127570e-17,False,0.000515,0.016800,0.000515,0.016800,True,STRUCTURE,B_CORE,RAW_A,1.127570e-15
29,EDU,GRAD_SCHOOL_PROGRESSION,fractional_logit,602,602,0.005316,0.005316,8.153200e-17,False,0.001640,0.012117,0.001640,0.012117,True,STRUCTURE,B_CORE,RAW_A,8.153200e-15
30,ENG,GRAD_SCHOOL_PROGRESSION,fractional_logit,911,911,0.045698,0.045698,4.163336e-17,False,0.016228,0.091214,0.016228,0.091214,True,STRUCTURE,B_CORE,RAW_A,4.163336e-15
31,HUM,GRAD_SCHOOL_PROGRESSION,fractional_logit,535,535,0.013732,0.013732,6.938894e-18,False,0.004090,0.031646,0.004090,0.031646,True,STRUCTURE,B_CORE,RAW_A,6.938894e-16
32,MED,GRAD_SCHOOL_PROGRESSION,fractional_logit,374,374,0.003801,0.003801,8.196568e-17,False,-0.000593,0.073071,-0.000593,0.073071,True,STRUCTURE,B_CORE,RAW_A,8.196568e-15
33,NAT,GRAD_SCHOOL_PROGRESSION,fractional_logit,583,583,0.036730,0.036730,0.000000e+00,False,0.011689,0.069876,0.011689,0.069876,True,STRUCTURE,B_CORE,RAW_A,0.000000e+00
34,SOC,GRAD_SCHOOL_PROGRESSION,fractional_logit,848,848,0.011690,0.011690,1.734723e-18,False,0.005174,0.023299,0.005174,0.023299,True,STRUCTURE,B_CORE,RAW_A,1.734723e-16
42,ART,HEALTH_EMPLOYMENT,fractional_logit,646,646,-0.004299,-0.004299,4.597017e-17,False,-0.014682,0.007325,-0.014682,0.007325,True,STRUCTURE,B_CORE,RAW_A,4.597017e-15
43,EDU,HEALTH_EMPLOYMENT,fractional_logit,602,602,0.015633,0.015633,5.551115e-17,False,0.000256,0.031026,0.000256,0.031026,True,STRUCTURE,B_CORE,RAW_A,5.551115e-15
44,ENG,HEALTH_EMPLOYMENT,fractional_logit,911,911,0.019294,0.019294,7.632783e-17,False,0.008278,0.028755,0.008278,0.028755,True,STRUCTURE,B_CORE,RAW_A,7.632783e-15


P5 strict residual status: PENDING_UPSTREAM_RESIDUAL


**P5 strict heterogeneity**

- 관찰: outcome별 양의 AME major 수는 {'GRAD_SCHOOL_PROGRESSION': 7, 'HEALTH_EMPLOYMENT': 4}이고, V1-vs-strict sign change는 0건이다.
- 원인: major7별 RAW_A 기울기는 outcome에 따라 방향과 크기가 다르며, strict 전환 민감도는 별도 축으로 확인해야 한다.
- 제한: AME는 조건부 association이고 residual branch는 아직 P3 residual 산출물 대기 상태다.
- 결론: `workbook/p2/p2_5/P2_G5_A2_visuals/figures/A2_08_p5_strict_heterogeneity.png`는 P5를 하나의 평균효과가 아니라 major/outcome별 이질성 문제로 읽게 한다.

## 8. Context 제한

P5 context는 7개 major point의 기술적 비교다. N=7이므로 context가 slope를 증가·감소시킨다는 confirmatory 문장을 쓰지 않는다.

### 구체 수치해석: context profile

- context 최댓값은 `ENG` ctx24_income_400plus=23.6%; `ENG` ctx24_large_mid_company=32.2%; `MED` goms_recent_firm_300plus=34.6%; `MED` goms_recent_permanent=88.4%; `MED` goms_recent_professional_highskill=88.6%다.
- 특히 ENG는 소득 400만 이상 비중(23.57%)과 대/중견기업 비중(32.20%)이 높고, MED는 300인 이상 기업(34.59%), 상용직(88.37%), 고숙련(88.56%)이 높다.
- 하지만 context는 major7 단위 7개 점뿐이다. 그래서 “context가 P5 slope 차이를 만든다”가 아니라 “slope를 읽을 때 같이 봐야 할 배경 조건”으로만 사용한다.

In [10]:
# context 변수는 major7 단위 기술통계다.
# slope의 원인 설명이 아니라, major별 노동시장/진학 맥락 차이를 함께 보는 보조판으로만 사용한다.
display(p5_context)
print("Context profile row count:", len(p5_context))
print("해석 제한: 계열 7개 점의 기술적 association만 보고한다. 인과 또는 meta-regression 결론은 금지한다.")

context_cols = [c for c in p5_context.columns if c != "major_group_7" and not c.endswith("__unique_n")]
context_matrix = p5_context.set_index("major_group_7")[context_cols].sort_index()
pretty_cols = {
    "ctx24_income_400plus_pct": "income 400+",
    "ctx24_large_mid_company_pct": "large/mid firm",
    "goms_recent_firm_300plus_pct": "firm 300+",
    "goms_recent_permanent_pct": "permanent",
    "goms_recent_professional_highskill_pct": "highskill",
}
context_matrix = context_matrix.rename(columns=pretty_cols)
fig, ax = plt.subplots(figsize=(10, 5.0))
heatmap(ax, context_matrix, "Major7 context profile (%)", cmap="YlGnBu", center_zero=False)
context_figure = savefig("A2_09_p5_context_profile.png")

high_context = context_matrix.stack().sort_values(ascending=False).head(3)
high_context_text = "; ".join([f"{idx[0]} {idx[1]}={val:.1f}%" for idx, val in high_context.items()])
visual_note(
    "Context 제한",
    f"가장 높은 context 셀은 {high_context_text}다.",
    "계열별 취업·소득·고용형태 맥락이 다르기 때문에 P5 slope를 해석할 때 배경판으로 함께 봐야 한다.",
    "major7은 7개 점뿐이라 context로 slope 차이를 통계적으로 설명하는 단계가 아니다.",
    f"`{context_figure}`는 confirmatory 분석이 아니라 해석 보조용 profile이다.",
)

,major_group_7,ctx24_income_400plus_pct,ctx24_income_400plus_pct__unique_n,ctx24_large_mid_company_pct,ctx24_large_mid_company_pct__unique_n,goms_recent_firm_300plus_pct,goms_recent_firm_300plus_pct__unique_n,goms_recent_permanent_pct,goms_recent_permanent_pct__unique_n,goms_recent_professional_highskill_pct,goms_recent_professional_highskill_pct__unique_n
0,ART,4.891948,1,12.847938,1,8.162268,1,63.28796,1,54.354107,1
1,EDU,5.142599,1,3.548845,1,4.769505,1,79.45341,1,82.351410,1
2,ENG,23.572332,1,32.198803,1,29.134941,1,84.83454,1,55.704210,1
3,HUM,16.769264,1,19.289959,1,21.097418,1,73.15509,1,27.957674,1
4,MED,13.206307,1,2.596486,1,34.586937,1,88.36591,1,88.556970,1
5,NAT,11.394269,1,21.980719,1,20.634125,1,74.28932,1,45.124050,1
6,SOC,16.076220,1,18.635672,1,20.332579,1,79.11846,1,23.923975,1


Context profile row count: 7
해석 제한: 계열 7개 점의 기술적 association만 보고한다. 인과 또는 meta-regression 결론은 금지한다.


**Context 제한**

- 관찰: 가장 높은 context 셀은 MED highskill=88.6%; MED permanent=88.4%; ENG permanent=84.8%다.
- 원인: 계열별 취업·소득·고용형태 맥락이 다르기 때문에 P5 slope를 해석할 때 배경판으로 함께 봐야 한다.
- 제한: major7은 7개 점뿐이라 context로 slope 차이를 통계적으로 설명하는 단계가 아니다.
- 결론: `workbook/p2/p2_5/P2_G5_A2_visuals/figures/A2_09_p5_context_profile.png`는 confirmatory 분석이 아니라 해석 보조용 profile이다.

## 9. 최종 판정과 인사이트 보드

P2-S는 strict-clean 기준으로 실행 완료됐다. P2-Q는 `selectivity_proxy_pct` feature approval이 열릴 때까지 차단한다. P5 strict 산출물은 strict hash와 manifest 기준으로 연결됐고, residual branch는 P3 이후 재실행 대상이다.

### 구체 수치해석: 최종 판정

- 현재 결론의 강한 축은 **P2-S strict 모델링 완료**와 **P5 RAW_A strict heterogeneity 연결 완료**다.
- P2-S는 개발 R2 12.6%까지 올라가지만 locked-test R2가 -6.7%라 예측 성능을 과장하면 안 된다. 이 노트북에서는 설명/진단 중심으로 읽는다.
- P2-Q는 `BLOCKED_FEATURE_CONTRACT` 상태다. `selectivity_proxy_pct` 승인 전까지 입결 관측 branch는 “보류”가 최종 판정이다.
- P5는 major/outcome별 이질성이 핵심이다. 진학률은 전 major 양수 방향, 건강취업률은 4개 양수/3개 음수 방향이라 결과변수별 해석을 분리해야 한다.
- 남은 실행 과제는 P3 residual 산출 이후 P5 residual branch 재실행, 그리고 P2-Q feature contract 승인 여부 결정이다.

In [11]:
# 마지막 셀은 표, 최종 상태 그래프, md 보고서, manifest를 동시에 갱신한다.
# 이 셀까지 실행돼야 A2 노트북의 시각화 산출물이 현재 상태로 닫힌다.
final_judgement = pd.DataFrame([
    {"item": "P2 strict D08", "status": p2_status.get("P2_INPUT_STATUS"), "note": p2_status.get("strict_d08_sha256")},
    {"item": "P2-S OLS", "status": p2_status.get("P2_S_OLS_STATUS"), "note": "P2-S4 R2={:.3f}".format(float(p2_specs.loc[p2_specs["model_id"].eq("P2-S4"), "r2"].iloc[0]))},
    {"item": "P2-Q OLS", "status": p2_status.get("P2_Q_OLS_STATUS"), "note": "selectivity_proxy_pct not manual-approved as feature"},
    {"item": "P2 GAM", "status": p2_status.get("P2_GAM_STATUS"), "note": "S4 spline checks available for structure variables"},
    {"item": "P2 MixedLM", "status": p2_status.get("P2_MIXEDLM_STATUS"), "note": "adjusted ICC={:.1f}% with warning".format(float(p2_mixed.loc[p2_mixed["model"].eq("ADJUSTED_S4"), "icc"].iloc[0]) * 100)},
    {"item": "P5 strict RAW_A", "status": p5_status.get("P5_RAW_A_STATUS"), "note": "major7 heterogeneity artifacts connected"},
    {"item": "P5 residual", "status": p5_status.get("P5_RESIDUAL_STATUS"), "note": "P3 residual pending"},
])
final_judgement["bucket"] = final_judgement["status"].map(status_bucket)
display(final_judgement)

fig, axes = plt.subplots(1, 2, figsize=(13.5, 4.2), gridspec_kw={"width_ratios": [1.0, 1.55]})
bucket_counts = final_judgement["bucket"].value_counts().reindex(["READY/OK", "WARN/PENDING", "BLOCKED", "OTHER", "UNKNOWN"]).dropna()
axes[0].bar(bucket_counts.index, bucket_counts.values, color=[{"READY/OK": PALETTE["green"], "WARN/PENDING": PALETTE["amber"], "BLOCKED": PALETTE["red"], "OTHER": PALETTE["muted"], "UNKNOWN": PALETTE["gray"]}.get(x, PALETTE["muted"]) for x in bucket_counts.index])
axes[0].set_title("Final judgement status")
axes[0].set_ylabel("count")
axes[0].tick_params(axis="x", rotation=20)
add_value_labels(axes[0], fmt="{:.0f}")

axes[1].axis("off")
final_figure_path = FIGURES_DIR / "A2_10_final_judgement_board.png"
projected_figure_count = len(set(FIGURES_DIR.glob("A2_*.png")) | {final_figure_path})
insight_lines = [
    f"P2-S4 dev R2: {float(p2_specs.loc[p2_specs['model_id'].eq('P2-S4'), 'r2'].iloc[0]) * 100:.1f}%",
    f"P2-S4 locked-test MAE: {float(p2_specs.loc[p2_specs['model_id'].eq('P2-S4'), 'test_mae'].iloc[0]):.2f}%p",
    f"P2-Q status: {p2_status.get('P2_Q_OLS_STATUS')}",
    f"P2 adjusted ICC: {float(p2_mixed.loc[p2_mixed['model'].eq('ADJUSTED_S4'), 'icc'].iloc[0]) * 100:.1f}%",
    f"P5 residual status: {p5_status.get('P5_RESIDUAL_STATUS')}",
    f"saved figures: {projected_figure_count}",
]
axes[1].text(0.02, 0.92, "Core modeling insights", fontsize=13, fontweight="bold", color=PALETTE["ink"])
for i, line in enumerate(insight_lines):
    axes[1].text(0.04, 0.78 - i * 0.12, f"- {line}", fontsize=10.5, color=PALETTE["ink"])
axes[1].set_title("Interpretation board", loc="left")
final_figure = savefig("A2_10_final_judgement_board.png")

final_figures = sorted(FIGURES_DIR.glob("A2_*.png"))
figure_rows = pd.DataFrame([{"figure": rel(p), "size_kb": round(p.stat().st_size / 1024, 1)} for p in final_figures])
display(figure_rows)

def markdown_table(df: pd.DataFrame) -> str:
    if df is None or df.empty:
        return "_empty_"
    out = df.copy().astype(object).where(pd.notna(df), "")
    cols = [str(c) for c in out.columns]
    lines = ["| " + " | ".join(cols) + " |", "| " + " | ".join(["---"] * len(cols)) + " |"]
    for _, row in out.iterrows():
        lines.append("| " + " | ".join(str(row[c]).replace("|", "/") for c in out.columns) + " |")
    return "\n".join(lines)

insight_report_lines = [
    "# P2-G5 A2 Visual Insight Notes",
    "",
    "## 핵심 인사이트",
    f"- P2-S4 개발 R2는 {float(p2_specs.loc[p2_specs['model_id'].eq('P2-S4'), 'r2'].iloc[0]) * 100:.1f}%이지만 locked-test MAE와 test R2를 별도로 확인해야 한다.",
    f"- P2-Q는 `{p2_status.get('P2_Q_OLS_STATUS')}` 상태라 현재 결론에서 제외한다.",
    f"- 입결 관측 표본 평균 A비율과 미관측 표본 차이는 {a_yes - a_no:.2f}%p다.",
    f"- 조정 MixedLM ICC는 {float(p2_mixed.loc[p2_mixed['model'].eq('ADJUSTED_S4'), 'icc'].iloc[0]) * 100:.1f}%로 학교 단위 분산이 남지만 warning을 동반한다.",
    f"- P5 residual branch는 `{p5_status.get('P5_RESIDUAL_STATUS')}`라 RAW_A strict heterogeneity와 분리한다.",
    "",
    "## 생성된 그림",
    markdown_table(figure_rows),
    "",
    "## 최종 판정",
    markdown_table(final_judgement[["item", "status", "bucket", "note"]]),
    "",
    "## 해석 가드레일",
    "- P2/P5 계수와 AME는 조건부 association이며 인과효과가 아니다.",
    "- P2-S와 P2-Q는 표본 관측 메커니즘이 다르므로 결론을 합치지 않는다.",
    "- P5 context는 major7의 기술통계 보조판이며 confirmatory meta-regression이 아니다.",
]
INSIGHT_PATH.write_text("\n".join(insight_report_lines), encoding="utf-8")

report_lines = [
    "# P2-G5 A2 Strict Visual Summary",
    "",
    "## Final judgement",
    markdown_table(final_judgement[["item", "status", "bucket", "note"]]),
    "",
    "## P2-S nested R2",
    markdown_table(p2_s_specs[["model_id", "status", "N", "school_n", "r2", "adj_r2", "test_mae", "test_r2"]]),
    "",
    "## P2-S key coefficients",
    markdown_table(coef_focus[["label", "coefficient", "ci_low", "ci_high", "p_value", "standardized_beta"]]),
    "",
    "## P5 primary RAW_A AME",
    markdown_table(p5_primary[["major_group_7", "outcome", "ame_pp_10pp", "ci_low_pp_10pp", "ci_high_pp_10pp"]]),
    "",
    "## Figures",
    markdown_table(figure_rows),
    "",
    "## Guardrails",
    "- P2/P5 coefficients are conditional associations, not causal effects.",
    "- P2-Q remains blocked under the current manual-approved feature contract.",
    "- P5 context analysis is descriptive only with seven major groups.",
]
REPORT_PATH.write_text("\n".join(report_lines), encoding="utf-8")

summary_manifest = {
    "created_at": datetime.now(timezone.utc).isoformat(),
    "notebook": artifact_row(THIS_NOTEBOOK, "P2_G5_A2 notebook"),
    "report": artifact_row(REPORT_PATH, "P2_G5_A2 summary report"),
    "visual_insight_notes": artifact_row(INSIGHT_PATH, "P2_G5_A2 visual insight notes"),
    "figures": [artifact_row(p, p.name) for p in final_figures],
    "inputs": lineage.to_dict("records"),
    "p2_status": {k: p2_status.get(k) for k in p2_status if k.startswith("P2_")},
    "p5_status": {k: p5_status.get(k) for k in p5_status if k.startswith("P5_")},
}
MANIFEST_PATH.write_text(json.dumps(summary_manifest, ensure_ascii=False, indent=2), encoding="utf-8")
print("report:", rel(REPORT_PATH))
print("manifest:", rel(MANIFEST_PATH))
print("visual insights:", rel(INSIGHT_PATH))
print("final figure:", final_figure)

visual_note(
    "최종 판정",
    f"최종 보드상 READY/OK {int((final_judgement['bucket'] == 'READY/OK').sum())}개, 차단/대기 {int((final_judgement['bucket'].isin(['BLOCKED', 'WARN/PENDING'])).sum())}개다.",
    "모델링 인사이트는 P2-S strict와 P5 RAW_A strict처럼 실행 가능한 branch에서만 작성하고, P2-Q/P5 residual은 제한으로 남겼다.",
    "보고서는 현재 실행 산출물을 요약하므로 upstream 재실행 후에는 이 노트북도 다시 실행해야 한다.",
    "A2의 결론은 시각화 board와 manifest/report/insight notes가 함께 저장된 상태로 닫힌다.",
)

,item,status,note,bucket
0,P2 strict D08,READY,5f56e375fd1c0474a5e55652859ae007e2f45becd6d335...,READY/OK
1,P2-S OLS,READY,P2-S4 R2=0.126,READY/OK
2,P2-Q OLS,BLOCKED_FEATURE_CONTRACT,selectivity_proxy_pct not manual-approved as f...,BLOCKED
3,P2 GAM,READY_WITH_WARNINGS,S4 spline checks available for structure varia...,WARN/PENDING
4,P2 MixedLM,MODEL_CONVERGENCE_WARNING,adjusted ICC=33.8% with warning,WARN/PENDING
5,P5 strict RAW_A,READY,major7 heterogeneity artifacts connected,READY/OK
6,P5 residual,PENDING_UPSTREAM_RESIDUAL,P3 residual pending,WARN/PENDING


,figure,size_kb
0,workbook/p2/p2_5/P2_G5_A2_visuals/figures/A2_0...,45.3
1,workbook/p2/p2_5/P2_G5_A2_visuals/figures/A2_0...,161.0
2,workbook/p2/p2_5/P2_G5_A2_visuals/figures/A2_0...,57.2
3,workbook/p2/p2_5/P2_G5_A2_visuals/figures/A2_0...,96.5
4,workbook/p2/p2_5/P2_G5_A2_visuals/figures/A2_0...,143.6
5,workbook/p2/p2_5/P2_G5_A2_visuals/figures/A2_0...,125.1
6,workbook/p2/p2_5/P2_G5_A2_visuals/figures/A2_0...,104.0
7,workbook/p2/p2_5/P2_G5_A2_visuals/figures/A2_0...,86.4
8,workbook/p2/p2_5/P2_G5_A2_visuals/figures/A2_0...,122.6
9,workbook/p2/p2_5/P2_G5_A2_visuals/figures/A2_0...,96.7


report: workbook/p2/p2_5/P2_G5_A2_SUMMARY_REPORT.md
manifest: workbook/p2/p2_5/P2_G5_A2_SUMMARY_MANIFEST.json
visual insights: workbook/p2/p2_5/P2_G5_A2_visuals/P2_G5_A2_VISUAL_INSIGHT_NOTES.md
final figure: workbook/p2/p2_5/P2_G5_A2_visuals/figures/A2_10_final_judgement_board.png


**최종 판정**

- 관찰: 최종 보드상 READY/OK 3개, 차단/대기 4개다.
- 원인: 모델링 인사이트는 P2-S strict와 P5 RAW_A strict처럼 실행 가능한 branch에서만 작성하고, P2-Q/P5 residual은 제한으로 남겼다.
- 제한: 보고서는 현재 실행 산출물을 요약하므로 upstream 재실행 후에는 이 노트북도 다시 실행해야 한다.
- 결론: A2의 결론은 시각화 board와 manifest/report/insight notes가 함께 저장된 상태로 닫힌다.